We're scraping the 'overlap' websites + course guides of the TU/e and CMU. 
https://tue.osiris-student.nl/onderwijscatalogus/extern/cursus?cursuscode=2IRR90&collegejaar=2025  
https://educationguide.tue.nl/programs/bachelor-college/elective-courses/overlap-courses  

https://www.cmu.edu/ini/academics/overlapping-course-list-5-31-24.pd 
https://www.courses.scottylabs.org/course/14-741  

In [15]:
# read csv with overlaps TU/e 
import pandas as pd 
filename = "/mimer/NOBACKUP/groups/naiss2024-22-903/WASP_NLP/project/data/TUE/overlap_table.csv"
df_overlap = pd.read_csv(filename)
df_overlap[65:73]
# forget about old course code, we'll just not be able to retrieve it and that's fine

,Course,New_code,Overlap
65,Security,2IRR40,2IC60
66,"Sets, Logic and Mathematic Language",2MBA10,2WF40
67,Statistics and Machine Learning,2IRR50,2IIG0 JBI030 2IIG5
68,Stochastic Optimization Models for Data Science,JBM100,2MBS30 2WB20
69,Stochastic Processes,2MBS30,2WB20 2WB29
70,Stochastic Simulation and Modeling,2MBS40,2WB50 2DI66
71,Stochastics and Simulation for Finance,2DF20,2WB50 2MBS40
72,Theory and Applications of Ordinary Differenti...,2MBC20,2WA70 2WA79


In [13]:
# Split the string, discard first item, join back
def discard_first_overlap(value):
    if not isinstance(value, str) or pd.isna(value):
        return value
    
    # Split by commas or spaces (or both)
    # Using regex to split on ', ' or ' ' or ','
    import re
    items = re.split(r'[, ]+', value.strip())
    
    # Discard first item, join back with comma
    return ','.join(items[1:]) if len(items) > 1 else ''

df_overlap['Overlap'] = df_overlap['Overlap'].apply(discard_first_overlap)
print(df_overlap)

                                               Course New_code        Overlap
0   Advance Simulation Methods for Ordinary Differ...   2MBC50               
1                                Algebraic Structures   2MBD20               
2                Algorithmic Aspects of Data Analysis   JBI045               
3                  Algorithmic Aspects of Game Theory   2IBA10            NaN
4                                          Analysis 1   2MBA40  JBM060,JBM080
..                                                ...      ...            ...
69                               Stochastic Processes   2MBS30          2WB29
70                 Stochastic Simulation and Modeling   2MBS40          2DI66
71             Stochastics and Simulation for Finance    2DF20         2MBS40
72  Theory and Applications of Ordinary Differenti...   2MBC20          2WA79
73                       Vector Analysis and Geometry   2MBC40          32VAN

[74 rows x 3 columns]


In [25]:
# Step 1: Just save the HTML (no parsing yet)
import re
import playwright
from playwright.async_api import async_playwright
async def save_html_only(url, output_path):
    async with async_playwright() as p:
        browser = await p.firefox.launch(headless=True)
        page = await browser.new_page()
        await page.goto(url, wait_until="domcontentloaded")
        await page.wait_for_timeout(3000)
        
        html = await page.content()
        with open(output_path, "w", encoding="utf-8") as f:
            f.write(html)
        
        await browser.close()
        print(f"Saved: {output_path}")

# Save multiple pages
output_path = '/mimer/NOBACKUP/groups/naiss2024-22-903/WASP_NLP/project/data/TUE/'
urls_and_files = [
    (df_overlap['New_code'][i], output_path+df_overlap['New_code'][i]+".html") for i in range(len(df_overlap))
    # Add more URLs
]
old_url = "https://tue.osiris-student.nl/onderwijscatalogus/extern/cursus?cursuscode=2IRR90&collegejaar=2025&taal=en"
for url, filename in urls_and_files:
    print(url)
    new_url =  re.sub(r'cursuscode=[^&]+', f'cursuscode={url}', old_url)
    await save_html_only(new_url, filename)

2MBC50
Saved: /mimer/NOBACKUP/groups/naiss2024-22-903/WASP_NLP/project/data/TUE/2MBC50.html
2MBD20
Saved: /mimer/NOBACKUP/groups/naiss2024-22-903/WASP_NLP/project/data/TUE/2MBD20.html
JBI045
Saved: /mimer/NOBACKUP/groups/naiss2024-22-903/WASP_NLP/project/data/TUE/JBI045.html
2IBA10
Saved: /mimer/NOBACKUP/groups/naiss2024-22-903/WASP_NLP/project/data/TUE/2IBA10.html
2MBA40
Saved: /mimer/NOBACKUP/groups/naiss2024-22-903/WASP_NLP/project/data/TUE/2MBA40.html
2MBA60
Saved: /mimer/NOBACKUP/groups/naiss2024-22-903/WASP_NLP/project/data/TUE/2MBA60.html
2MBD30
Saved: /mimer/NOBACKUP/groups/naiss2024-22-903/WASP_NLP/project/data/TUE/2MBD30.html
2IRR90
Saved: /mimer/NOBACKUP/groups/naiss2024-22-903/WASP_NLP/project/data/TUE/2IRR90.html
JBE140
Saved: /mimer/NOBACKUP/groups/naiss2024-22-903/WASP_NLP/project/data/TUE/JBE140.html
2WBB0
Saved: /mimer/NOBACKUP/groups/naiss2024-22-903/WASP_NLP/project/data/TUE/2WBB0.html
2WCB0
Saved: /mimer/NOBACKUP/groups/naiss2024-22-903/WASP_NLP/project/data/TUE/2WC

In [34]:
# now also scrape the overlapping courses (in a dict?) 
# find all unique ones that are not in the 'new code' column 
# keep in mind that it's possible that the URL fails. Save the failed urls? 

# find unique ones 
all_overlap_codes = set()
for overlap_str in df_overlap['Overlap'].dropna():
    # Split by spaces to get individual codes
    codes = str(overlap_str).split()
    all_overlap_codes.update(codes)

# Step 2: Get all codes from New_code column
new_codes = set(df_overlap['New_code'].dropna().astype(str).str.strip())

# Step 3: Find overlap codes that are NOT in new_codes
codes_to_scrape = all_overlap_codes - new_codes

print(f"Step 3: Overlap codes NOT in New_code:")
print(f"Count: {len(codes_to_scrape)}")
#print(f"Codes: {sorted(codes_to_scrape)}")

Step 3: Overlap codes NOT in New_code:
Count: 99


In [40]:
output_path = '/mimer/NOBACKUP/groups/naiss2024-22-903/WASP_NLP/project/data/TUE/'
old_url = "https://tue.osiris-student.nl/onderwijscatalogus/extern/cursus?cursuscode=2IRR90&collegejaar=2025&taal=en"

failed_codes = [] # some of the codes are old course codes 
for code in codes_to_scrape: 
    print(code)
    new_url =  re.sub(r'cursuscode=[^&]+', f'cursuscode={code}', old_url)
    filename = output_path+str(code)+".html"
    try: 
        await save_html_only(new_url, filename)
    except Exception as e: 
        print(e)
        failed_codes.append(code)
print(len(failed_codes))

2DBN10
Saved: /mimer/NOBACKUP/groups/naiss2024-22-903/WASP_NLP/project/data/TUE/2DBN10.html
2WF70
Saved: /mimer/NOBACKUP/groups/naiss2024-22-903/WASP_NLP/project/data/TUE/2WF70.html
2DD80
Saved: /mimer/NOBACKUP/groups/naiss2024-22-903/WASP_NLP/project/data/TUE/2DD80.html
2ITS60
Saved: /mimer/NOBACKUP/groups/naiss2024-22-903/WASP_NLP/project/data/TUE/2ITS60.html
2WA30
Saved: /mimer/NOBACKUP/groups/naiss2024-22-903/WASP_NLP/project/data/TUE/2WA30.html
2DD55
Saved: /mimer/NOBACKUP/groups/naiss2024-22-903/WASP_NLP/project/data/TUE/2DD55.html
2WF50
Saved: /mimer/NOBACKUP/groups/naiss2024-22-903/WASP_NLP/project/data/TUE/2WF50.html
2DL70
Saved: /mimer/NOBACKUP/groups/naiss2024-22-903/WASP_NLP/project/data/TUE/2DL70.html
0LVX30
Saved: /mimer/NOBACKUP/groups/naiss2024-22-903/WASP_NLP/project/data/TUE/0LVX30.html
6A6X0
Saved: /mimer/NOBACKUP/groups/naiss2024-22-903/WASP_NLP/project/data/TUE/6A6X0.html
2IHA10
Saved: /mimer/NOBACKUP/groups/naiss2024-22-903/WASP_NLP/project/data/TUE/2IHA10.html
2W

Future exception was never retrieved
future: <Future finished exception=TargetClosedError('Target page, context or browser has been closed')>
playwright._impl._errors.TargetClosedError: Target page, context or browser has been closed
Future exception was never retrieved
future: <Future finished exception=TargetClosedError('Target page, context or browser has been closed')>
playwright._impl._errors.TargetClosedError: Target page, context or browser has been closed
Future exception was never retrieved
future: <Future finished exception=TargetClosedError('Target page, context or browser has been closed')>
playwright._impl._errors.TargetClosedError: Target page, context or browser has been closed


Saved: /mimer/NOBACKUP/groups/naiss2024-22-903/WASP_NLP/project/data/TUE/2WN20.html
8BB020
Saved: /mimer/NOBACKUP/groups/naiss2024-22-903/WASP_NLP/project/data/TUE/8BB020.html
2WA70
Saved: /mimer/NOBACKUP/groups/naiss2024-22-903/WASP_NLP/project/data/TUE/2WA70.html
JBI025
Saved: /mimer/NOBACKUP/groups/naiss2024-22-903/WASP_NLP/project/data/TUE/JBI025.html
1BK60
Saved: /mimer/NOBACKUP/groups/naiss2024-22-903/WASP_NLP/project/data/TUE/1BK60.html
2WAH0
Saved: /mimer/NOBACKUP/groups/naiss2024-22-903/WASP_NLP/project/data/TUE/2WAH0.html
1BV30
Saved: /mimer/NOBACKUP/groups/naiss2024-22-903/WASP_NLP/project/data/TUE/1BV30.html
+
Saved: /mimer/NOBACKUP/groups/naiss2024-22-903/WASP_NLP/project/data/TUE/+.html
2ITS60;
Saved: /mimer/NOBACKUP/groups/naiss2024-22-903/WASP_NLP/project/data/TUE/2ITS60;.html
2WA80
Saved: /mimer/NOBACKUP/groups/naiss2024-22-903/WASP_NLP/project/data/TUE/2WA80.html
JBI025;
Saved: /mimer/NOBACKUP/groups/naiss2024-22-903/WASP_NLP/project/data/TUE/JBI025;.html
2WF30
Saved:

In [ ]:
# manually input all old codes 
old_codes = ['2WN30', '2WF50', 'JBI040', '2WA30', 'JBM060', '2WF70', '2IT90', '2ILH0', 
             '2WA80', 'JBM020', 'JBM010', 'JBI025', 'JBI020', 'JBM090', 'JBM090-B-5', '2IAB0', '2WS30', '2WF60', 
             '2WF80', '2WN20', 'JBL120', '2WF20', '2WF20', '2WF30', '2DBI00', '2DRR00', 'JBM070', '2WO020', 
             '2WH60', '2WAG0', '2IC60', '2WN40', '2WA90', '2WS20', '2WH20', '2WB40', '2WS40', '2IC60', '2WF40', 
             '2IIG0', '2WB20', '2WB50', '2WA70', '2WA60'] 
print(len(old_codes))

In [42]:
### Now try to parse the HTML files as JSON using their structure 
from bs4 import BeautifulSoup
import json

def parse_html_auto(html_file):
    """Extract structured data from ANY HTML file automatically"""
    with open(html_file, 'r', encoding='utf-8') as f:
        soup = BeautifulSoup(f.read(), 'html.parser')
    
    # Remove non-content elements
    for tag in soup(["script", "style", "nav", "footer", "header", "aside"]):
        tag.decompose()
    
    result = {
        'title': soup.title.string.strip() if soup.title else '',
        'headings': [h.get_text().strip() for h in soup.find_all(['h1', 'h2', 'h3']) if h.get_text().strip()],
        'paragraphs': [p.get_text().strip() for p in soup.find_all('p') if p.get_text().strip()],
        'lists': [],
        'tables': [],
        'full_text': soup.get_text()
    }
    
    # Extract list items
    for ul in soup.find_all(['ul', 'ol']):
        items = [li.get_text().strip() for li in ul.find_all('li') if li.get_text().strip()]
        if items:
            result['lists'].append(items)
    
    # Extract tables
    for table in soup.find_all('table'):
        rows = []
        for tr in table.find_all('tr'):
            cells = [td.get_text().strip() for td in tr.find_all(['td', 'th']) if td.get_text().strip()]
            if cells:
                rows.append(cells)
        if rows:
            result['tables'].append(rows)
    
    return result

# Use it
html_file = '/mimer/NOBACKUP/groups/naiss2024-22-903/WASP_NLP/project/data/TUE/'+"2IRR50"+".html"
data = parse_html_auto(html_file)
print(json.dumps(data, indent=2))

{
  "title": "Osiris Student Mobile",
  "headings": [],
  "paragraphs": [],
  "lists": [
    [
      "Block 4Timeslot(s)A - Mo 1-4, We 9-10, Th 5-8Enrolment period15 November 2025 until 22 March 2026 23:59Unenrolment until 20 April 2026 23:59"
    ],
    [
      "Completed none of the course modules listed belowData Mining and Machine Learning (2IIG0)",
      "Data Mining and Machine Learning (2IIG0)"
    ],
    [
      "Data Mining and Machine Learning (2IIG0)"
    ],
    [
      "InstructionRemark",
      "LectureRemark"
    ],
    [
      "Assignment 1Interim examinationTest weight20Minimum grade",
      "Assignment 2Interim examinationTest weight20Minimum grade",
      "Written examination using notebookFinal examinationTest weight60Minimum grade5.0"
    ],
    [
      "Block 4Enrolment period15 November 2025 until 7 June 2026 23:59"
    ],
    [
      "Block 4Enrolment period15 November 2025 until 7 June 2026 23:59"
    ],
    [
      "Block 4Enrolment period15 November 2025 until

In [43]:
# now we're ready to save it and process it using an LLM ! 
import os
uni_dir = '/mimer/NOBACKUP/groups/naiss2024-22-903/WASP_NLP/project/data/TUE/' 
all_files = os.listdir(uni_dir)
html_files = [f for f in all_files if f.lower().endswith('.html')] 
for html_file in html_files: 
    # Create JSON filename by replacing .html with .json
    json_file = uni_dir + html_file.replace('.html', '.json')
    data = parse_html_auto(uni_dir + html_file)
    # Save to JSON file
    with open(json_file, 'w', encoding='utf-8') as f:
        json.dump(data, f, indent=2, ensure_ascii=False)
    
    print(f"Saved to: {json_file}")

Saved to: /mimer/NOBACKUP/groups/naiss2024-22-903/WASP_NLP/project/data/TUE/2IRR40.json
Saved to: /mimer/NOBACKUP/groups/naiss2024-22-903/WASP_NLP/project/data/TUE/2MBA90.json
Saved to: /mimer/NOBACKUP/groups/naiss2024-22-903/WASP_NLP/project/data/TUE/2WS40.json
Saved to: /mimer/NOBACKUP/groups/naiss2024-22-903/WASP_NLP/project/data/TUE/2DL15.json
Saved to: /mimer/NOBACKUP/groups/naiss2024-22-903/WASP_NLP/project/data/TUE/2IRR10.json
Saved to: /mimer/NOBACKUP/groups/naiss2024-22-903/WASP_NLP/project/data/TUE/JBI025;.json
Saved to: /mimer/NOBACKUP/groups/naiss2024-22-903/WASP_NLP/project/data/TUE/2WB50.json
Saved to: /mimer/NOBACKUP/groups/naiss2024-22-903/WASP_NLP/project/data/TUE/2IRR20.json
Saved to: /mimer/NOBACKUP/groups/naiss2024-22-903/WASP_NLP/project/data/TUE/6E5X0.json
Saved to: /mimer/NOBACKUP/groups/naiss2024-22-903/WASP_NLP/project/data/TUE/JBM035.json
Saved to: /mimer/NOBACKUP/groups/naiss2024-22-903/WASP_NLP/project/data/TUE/2MBC40.json
Saved to: /mimer/NOBACKUP/groups/na